In [1]:
import torch
from torch import nn
from torch.nn import functional as F 

In [2]:
net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
#                  net[0]         net[1]    net[2]
x=torch.rand(2,4)
net(x)

tensor([[0.2237],
        [0.2068]], grad_fn=<AddmmBackward0>)

In [3]:
print(net[2].state_dict())
#use state_dict()to check the weights of the layer

OrderedDict({'weight': tensor([[-0.0552, -0.3300,  0.0816, -0.2892,  0.1173,  0.1621, -0.0564,  0.1567]]), 'bias': tensor([0.3038])})


In [10]:
net[2].weight.data[0,0]=.114514  #we can set the specific weight/bias number  
                              #bias dim=1,while weight dim=2 so different expression
                              #weight/bias=data+grad
net[2].bias.data[0]=.1919810         
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.1145, -0.3300,  0.0816, -0.2892,  0.1173,  0.1621, -0.0564,  0.1567]]), 'bias': tensor([0.1920])})


In [5]:
# we can also print one specific parameter
print(type(net[2].bias))
print(net[2].bias)      #have two: data, grad
print(net[2].bias.data)
print(net[2].bias.grad==None)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.2829], requires_grad=True)
tensor([-0.2829])
True


In [6]:
net[2].weight.grad==None

True

check all the parameters

In [7]:
print(*[(name,param.shape) for name,param in net[0].named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))


In [8]:
print(*[(name,param.shape) for name,param in net.named_parameters()])

('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [9]:
net.state_dict()['2.bias'].data
#state_dict is a list,so [] to check

tensor([-0.2829])

initialize the parameter yourself

In [10]:
#set parameters as normal value
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)  #replace itself,no return
        nn.init.zeros_(m.bias)  #same
net.apply(init_normal)   #for every module in net ,call init_normal  
net[0].weight.data[0],net[0].bias.data[0]

(tensor([-0.0069,  0.0044,  0.0069,  0.0030]), tensor(0.))

In [11]:
#set parameters as a constant value(no meanning)
def init_constant(m):
    if type(m)==nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

shared weights


In [18]:
shared=nn.Linear(20,20)
net=nn.Sequential(nn.Linear(4,20),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(20,1))
net(x)

print(net[2].weight.data[0,0])
net[2].weight.data[0,0]=100
print(net[4].weight.data[0,0])

tensor(0.1042)
tensor(100.)


self try multiple layer weight set:

In [11]:
net1=nn.Sequential(nn.Linear(10,256),nn.ReLU(),nn.Linear(256,20),nn.ReLU(),nn.Linear(20,5))
net1(torch.rand(4,10))

tensor([[-0.1156, -0.1357,  0.1797, -0.1452, -0.0787],
        [-0.1039, -0.1816,  0.1812, -0.1651, -0.0790],
        [-0.1153, -0.1540,  0.1530, -0.1783, -0.1212],
        [-0.1445, -0.1864,  0.1600, -0.1777, -0.1624]],
       grad_fn=<AddmmBackward0>)

In [14]:
net1[4].weight.data[0,1]=0.123
print(net1[4].weight.data)

tensor([[-0.1869,  0.1230, -0.0910, -0.0586,  0.0209,  0.2025,  0.1151,  0.0408,
         -0.1598,  0.0294, -0.1488,  0.0842,  0.0360,  0.2075, -0.1651,  0.0461,
          0.0630,  0.2234, -0.1033, -0.1121],
        [-0.1683, -0.0326,  0.1738,  0.2219,  0.1208,  0.1955,  0.0223,  0.0678,
         -0.0402, -0.0531, -0.1451,  0.0378, -0.0981, -0.0187,  0.0340,  0.0288,
         -0.2100, -0.2087, -0.1434, -0.1246],
        [ 0.1114,  0.0664, -0.0639,  0.1931, -0.0359, -0.0565, -0.0523,  0.0126,
         -0.1322,  0.1750,  0.1617,  0.0660, -0.1430,  0.1068, -0.0275, -0.0147,
          0.0168, -0.0221, -0.1943,  0.2131],
        [-0.0556, -0.0223, -0.1241,  0.0210, -0.0332, -0.1168,  0.2165,  0.2087,
         -0.0664,  0.1183, -0.2110, -0.1678,  0.1228, -0.1568, -0.0132, -0.1964,
          0.1871,  0.0582, -0.0666,  0.1770],
        [ 0.2051, -0.1292,  0.1015,  0.1130,  0.0168, -0.0542, -0.0584, -0.1793,
          0.0424,  0.0568, -0.1905,  0.0086, -0.1378,  0.1782,  0.0256, -0.0493,
      